In [1]:
%pip install -q geopandas rasterstats rasterio rioxarray shapely pyogrio tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import geopandas as gpd

districts = gpd.read_file("./Data/gadm41_IND_shp/gadm41_IND_2.shp")

print(districts.columns)
print(districts.head())

Index(['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2',
       'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2',
       'geometry'],
      dtype='str')
       GID_2 GID_0 COUNTRY    GID_1               NAME_1 NL_NAME_1  \
0  IND.1.1_1   IND   India  IND.1_1  Andaman and Nicobar        NA   
1  IND.1.2_1   IND   India  IND.1_1  Andaman and Nicobar        NA   
2  IND.1.3_1   IND   India  IND.1_1  Andaman and Nicobar        NA   
3  IND.2.1_1   IND   India  IND.2_1       Andhra Pradesh        NA   
4  IND.2.2_1   IND   India  IND.2_1       Andhra Pradesh        NA   

                     NAME_2             VARNAME_2 NL_NAME_2    TYPE_2  \
0           Nicobar Islands                    NA        NA  District   
1  North and Middle Andaman                    NA        NA  District   
2             South Andaman                    NA        NA  District   
3                 Anantapur  Anantpur, Ananthapur        NA  District   
4                  Chit

In [ ]:
districts = districts.to_crs("EPSG:4326")

districts["centroid"] = districts.geometry.centroid

districts["latitude"] = districts.centroid.y
districts["longitude"] = districts.centroid.x

In [3]:
from rasterstats import zonal_stats

def extract_soil_property(
        districts,
        raster_path,
        column_name):

    stats = zonal_stats(
        districts,
        raster_path,
        stats=["mean"],
        nodata=-9999
    )

    districts[column_name] = [
        s["mean"] for s in stats
    ]

    return districts

In [6]:
from rasterstats import zonal_stats

sample = districts.iloc[:5]

stats = zonal_stats(
    sample,
    "./Data/Soil/Nitrogen.tif",
    stats=["mean"]
)

for i, stat in enumerate(stats):
    print(
        sample.iloc[i]["NAME_2"],
        stat["mean"]
    )

d:\Agrisense\agrisense-backend\.venv\Lib\site-packages\rasterstats\io.py:437: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Nicobar Islands 335.30200778685395
North and Middle Andaman 342.7717147100263
South Andaman 368.78851876234364
Anantapur 133.66527332698678
Chittoor 190.13633900879262


In [5]:
import rasterio

with rasterio.open(
    "./Data/Soil/Nitrogen.tif"
) as src:

    print(src.crs)
    print(src.bounds)
    print(src.width, src.height)

EPSG:4326
BoundingBox(left=68.0, bottom=6.0, right=98.00000000000001, top=38.0)
13271 13390


In [ ]:
import requests

def get_soil_data(lat, lon):

    properties = [
        "phh2o",
        "nitrogen",
        "soc",
        "cec",
        "clay",
        "sand",
        "silt"
    ]

    params = {
        "lon": lon,
        "lat": lat,
        "property": properties,
        "depth": ["0-5cm"],
        "value": ["mean"]
    }

    url = "https://rest.isric.org/soilgrids/v2.0/properties/query"

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return None

    return response.json()

In [ ]:
sample = districts[
    districts["NAME_1"] == "Karnataka"
].iloc[0]

soil = get_soil_data(
    sample.latitude,
    sample.longitude
)

print(soil)

In [ ]:
print(len(districts))

print(
    districts[
        ["NAME_1","NAME_2"]
    ].head()
)

In [ ]:
districts["rep_point"] = districts.geometry.representative_point()

districts["latitude"] = districts["rep_point"].y
districts["longitude"] = districts["rep_point"].x

sample = districts.iloc[100]

print(sample["NAME_1"])
print(sample["NAME_2"])
print(sample["latitude"])
print(sample["longitude"])

In [ ]:
soil = get_soil_data(
    26.132385255000116,
    84.38311788118521
)

import json
print(json.dumps(soil, indent=2))

In [ ]:
def parse_soil(response):

    result = {}

    for layer in response["properties"]["layers"]:

        name = layer["name"]

        d_factor = layer["unit_measure"]["d_factor"]

        raw_value = layer["depths"][0]["values"]["mean"]

        if raw_value is None:
            result[name] = None
        else:
            result[name] = raw_value / d_factor

    return result

In [ ]:
import requests

def get_soil_data(lat, lon):

    properties = [
        "phh2o",
        "nitrogen",
        "soc",
        "cec",
        "clay",
        "sand",
        "silt"
    ]

    params = {
        "lon": lon,
        "lat": lat,
        "property": properties,
        "depth": ["0-5cm"],
        "value": ["mean"]
    }

    url = "https://rest.isric.org/soilgrids/v2.0/properties/query"

    response = requests.get(url, params=params, timeout=30)

    response.raise_for_status()

    return response.json()

In [ ]:
def parse_soil(response):

    result = {}

    for layer in response["properties"]["layers"]:

        name = layer["name"]

        d_factor = layer["unit_measure"]["d_factor"]

        raw_value = layer["depths"][0]["values"]["mean"]

        if raw_value is None:
            result[name] = None
        else:
            result[name] = raw_value / d_factor

    return result

In [ ]:
def process_district(row):

    try:

        soil_json = get_soil_data(
            row["latitude"],
            row["longitude"]
        )

        soil = parse_soil(soil_json)

        return {
            "State_Name": row["NAME_1"],
            "District_Name": row["NAME_2"],
            "PH": soil.get("phh2o"),
            "Clay": soil.get("clay"),
            "Sand": soil.get("sand"),
            "Silt": soil.get("silt"),
            "Nitrogen": soil.get("nitrogen"),
            "SOC": soil.get("soc"),
            "CEC": soil.get("cec")
        }

    except Exception as ex:

        print(
            f"Failed : {row['NAME_1']} - {row['NAME_2']} : {ex}"
        )

        return None

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd

rows = []

district_records = districts.to_dict("records")

MAX_WORKERS = 15

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [
        executor.submit(process_district, row)
        for row in district_records
    ]

    for future in tqdm(
            as_completed(futures),
            total=len(futures)):

        result = future.result()

        if result:
            rows.append(result)

soil_df = pd.DataFrame(rows)

In [ ]:
soil_df = pd.DataFrame(rows)

soil_df.to_csv(
    "india_district_soil_profile.csv",
    index=False
)